# Dual Use Calculation Optimizer

Rebuilds the `Difference` tab of a Dual Use Calculations workbook from its two source tabs, in
Python, and writes a new workbook carrying both source tabs plus the rebuilt tab as
`Difference_calc`.

Plug and play: put the workbook in a folder, set that folder in Section 2, Run All.

## Input requirements

**Requirement 1. File name.** The input workbook must be named:

```
Dual_Use_Calculations_TwoAge_B01_<Product>_<Month>_<Day>_<Year>.xlsx
```

for example `Dual_Use_Calculations_TwoAge_B01_Grizzly_XL_August_31_2026.xlsx`. `<Product>` is free
text, so `Grizzly_XL`, `Velo_Max`, `Glo` and `Vuse` all work. The product and the run date are read
straight out of the name, which is why nothing has to be typed twice.

**Requirement 2. Two tabs in, one tab out.** The workbook must carry a `Detailed Results` tab and a
`MasterModels TP-ID` tab. Those two generate the third, `Difference`. Tab names are matched loosely,
so `Raw Detailed Results` satisfies the first and `MasterModels TP-1D` the second: the digit one and
the letter I are indistinguishable in most fonts, so both spellings are accepted.

**Requirement 3. Model Group.** `Model Group` must read `<Product>_<your initials>`, for example
`Grizzly_XL_YO`. The product taken from it is checked against the product in the file name, and the
initials are recorded on the output.

## What the two tabs contribute

| Tab | Role |
|---|---|
| `Detailed Results` | The dual use model runs. Four ERRs per gateway per sex: two exclusive use values (0.05, 0.10) and two dual use values (1.05, 1.10). These are the survivor differences the interpolation runs between |
| `MasterModels TP-ID` | The no quitting baseline, one mean per gateway, sex and exclusive use ERR. Feeds the `No quitting` column and the `% change` beside it |

## Output

One workbook, `Dual_Use_Calculations_TwoAge_B01_<Product>_<YYYYMMDD>.xlsx`, with three tabs: the two
source tabs carried through unchanged, and `Difference_calc`.

## Sections

| Section | What it does | Why it is there |
|---|---|---|
| 1 | Imports and layout constants | Every position and value that defines the tab, in one place |
| 2 | Configuration | The only cell that changes between runs |
| 3 | Requirements 1, 2 and 3 | Fails early and by name rather than deep inside the build |
| 4 | Reading the two tabs | Normalises both into one lookup shape |
| 5 | The rules and formulae | The complete catalogue of what the tab does, and the layout engine |
| 6 | Building `Difference_calc` | Writes the cells, live formulas and all |
| 7 | Writing the workbook | Three tabs, formats applied |
| 8 | Orchestrator and run | One call end to end |
| 9 | Validation | Rebuilt values checked cell by cell against the source tab |
| 10 | Troubleshooting and known differences | Every error, and every place the rebuild deliberately differs |

## The `Difference` tab, rule by rule

The tab is two near identical blocks, one per gateway, 22 rows apart: G10 starts at row 1, G25 at
row 23. Everything below is expressed relative to a block's first row, `base`.

### 1. Left table, the model results (columns A to D)

| Cell | Content | Rule |
|---|---|---|
| `A{base}:D{base}` | `Model Name`, `ERR`, `Node`, `Mean` | Column headers |
| `E{base}` | `0.10` or `0.25` | Gateway effect as a fraction, read from the `G10` or `G25` token in the model name, shown as `0%` |
| `F{base}` | `Male` | Marks which sex the first dual use table belongs to |
| `A{base+1}`, `A{base+6}` | `Male`, `Female` | Section labels |
| `A{base+2}` to `A{base+5}` | The four male rows | From `Detailed Results`, this gateway, ERR ascending: 0.05, 0.10, 1.05, 1.10 |
| `A{base+7}` to `A{base+10}` | The four female rows | Same, female |

The two low ERRs are the exclusive use runs and the two high ERRs the dual use runs. That pairing is
what the rest of the tab interpolates between.

### 2. Dual use interpolation tables (columns F to R)

Four tables per block, one per sex per exclusive use ERR, each two rows deep:

| Cell | Formula in the source | Rule |
|---|---|---|
| `H{hdr}:R{hdr}` | `0, 0.1, ... 1.0` | The % dual use axis, eleven steps |
| `F`, `G` | ERR A, ERR B | ERR A is the exclusive use ERR, written once per pair; ERR B is 1.05 or 1.10 |
| `H{r}` | `=D{row of ERR A}` | Left end, 0% dual use, is the exclusive use mean |
| `R{r}` | `=D{row of ERR B}` | Right end, 100% dual use, is the dual use mean |
| `I{r}` to `Q{r}` | `=(1-I$12)*$H4+I$12*$R4` | Straight line interpolation, `value = (1 - w) x ERR A mean + w x ERR B mean` |

So each row is the survivor difference as the share of dual users runs from none to all.

### 3. No quitting comparison (columns B to D, below the left table)

| Cell | Formula in the source | Rule |
|---|---|---|
| `C{base+13}`, `D{base+13}` | `No quitting`, `% change` | Headers |
| `B` | `M`, then 0.05 and 0.10, then `F`, then 0.05 and 0.10 | Sex and exclusive use ERR |
| `C` | value | The `MasterModels TP-ID` mean for that gateway, sex and ERR, the no quitting baseline |
| `D` | `=(D3-C16)/D3` | `(dual use mean - no quitting mean) / dual use mean`, shown as `0%` |

### 4. Chart feed (columns T to AD, and AF to AG)

The source pasted two interpolation rows as values and transposed them for the chart. Only the
matched pairs are used, ERR A 0.05 with ERR B 1.05, and 0.10 with 1.10, male only.

| Cell | Rule |
|---|---|
| `T{base+3}:AD{base+3}` | Row `base+3` of the male table, as values |
| `T{base+7}:AD{base+7}` | Row `base+7`, as values |
| `AF{base+3}:AF{base+13}` | The first of those two, transposed into a column |
| `AG{base+3}:AG{base+13}` | The second, transposed |

### 5. Tipping points (columns J to N, below the last block)

| Cell | Formula in the source | Rule |
|---|---|---|
| `K{h}:N{h}` | `ERR A/ERR B`, `Gateway`, `Used Product in Combination with Cigarettes`, `Used Product Exclusively` | Headers |
| `M` | `=M3+0.1*M4/(M4+ABS(N4))` | The % dual use at which the interpolated difference crosses zero |
| `N` | `=1-M44` | The remainder, the exclusively exclusive share |

The M formula is linear interpolation between the two % dual use steps that bracket the sign change:
`tipping point = w_low + step x v_low / (v_low + |v_high|)`. In the source the bracketing pair is
hard coded per row, and correctly so: three rows cross between 0.5 and 0.6 and one between 0.4 and
0.5, and each formula points at its own pair. This rebuild finds the crossing pair by looking at the
values, then writes the formula against those columns, which gives the same answer here and does not
have to be re-pointed by hand when the numbers move.

## Section 1. Imports and layout constants

**Rationale.** The tab is a fixed geometry: two blocks 22 rows apart, four interpolation tables per
block at known offsets, eleven % dual use steps. Putting every one of those numbers in one place is
what makes the rebuild readable and what makes a layout change a one line edit rather than a hunt
through the writer.

The offsets are all expressed relative to the block's first row, so the second block is the first
plus `BLOCK_STRIDE` and nothing else has to know there is more than one gateway.

In [ ]:
# %pip install pandas numpy openpyxl

from __future__ import annotations

import re
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter as L

# ------------------------------------------------------- Requirement 1, names
INPUT_REGEX = re.compile(
    r"^Dual_Use_Calculations_TwoAge_B01_(?P<product>.+)_"
    r"(?P<month>[A-Za-z]+)_(?P<day>\d{1,2})_(?P<year>\d{4})\.xlsx$")
OUTPUT_TEMPLATE = "Dual_Use_Calculations_TwoAge_B01_{product}_{stamp}.xlsx"

# ------------------------------------------------- Requirement 2, tab matching
# Matched on the letters and digits only, so 'Raw Detailed Results' finds the first and both
# 'MasterModels TP-ID' and 'MasterModels TP-1D' find the second.
SHEET_KEYS = {"detailed": ("detailedresults",), "master": ("mastermodels",)}
OUTPUT_DIFF_SHEET = "Difference_calc"
SOURCE_DIFF_SHEET = "Difference"          # only read, for the Section 9 validation

# --------------------------------------------------------------- model layout
ERR_A_VALUES = [0.05, 0.10]               # exclusive use runs
ERR_B_VALUES = [1.05, 1.10]               # dual use runs
MATCHED_PAIRS = [(0.05, 1.05), (0.10, 1.10)]   # the pairs the chart and tipping points use
SEX_ORDER = ["Male", "Female"]
GATEWAY_ORDER = ["G10", "G25"]
GATEWAY_FRACTION = {"G10": 0.10, "G25": 0.25}  # the value written to E{base}, shown as 0%
DUAL_USE_STEPS = [round(0.1 * i, 10) for i in range(11)]   # 0, 0.1, ... 1.0
TP_SEXES = ["Male"]                       # the source tipping point table is male only

# ----------------------------------------------------------- sheet geometry
BLOCK_STRIDE = 22                 # G10 at row 1, G25 at row 23
SEX_TABLE_STRIDE = 9              # male table header at base+2, female at base+11
COL = {"H": 8, "R": 18, "T": 20, "AD": 30, "AF": 32, "AG": 33}
OFF = {                           # every offset from a block's first row
    "left_header": 0, "sex_label_1": 1, "sex_rows_1": 2, "sex_label_2": 6, "sex_rows_2": 7,
    "dual_label": 0, "dual_pct_label": 1, "dual_header": 2, "dual_first": 3,
    "nq_header": 13, "nq_first": 14, "tp_offset": 19,
}

# --------------------------------------------------------------------- style
FONT = "Calibri"
FONT_SIZE = 11
FILL_HEADER = "ED7D31"            # theme accent 2, the orange header fill in the source
FILL_SEX = "00B0F0"              # the blue sex labels in the source
FILL_GATEWAY = "FFFF00"          # the yellow gateway cell in the source
FMT_COUNT = "#,##0"
FMT_PCT = "0%"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print(f"pandas {pd.__version__} | numpy {np.__version__}")

## Section 2. Configuration

**Rationale.** One folder. The product, the date and the tab names all come from the input, so there
is nothing else to keep in sync. `INPUT_FILE = None` picks up the single matching workbook in the
folder, which is what makes this plug and play; name a file explicitly when the folder holds several.

In [ ]:
# The folder holding the Dual Use Calculations workbook. Use a raw string (r"...") on Windows.
DATA_DIR = r"C:\Users\YemiOdeyemi\Downloads\Grizzly\Dual_Use"
OUT_DIR = None            # None writes the output next to the input

INPUT_FILE = None         # None finds the one file matching the Requirement 1 pattern

# The stamp appended to the output file name. "%Y%m%d" gives 20260904.
OUTPUT_DATE_FORMAT = "%Y%m%d"

# Written into the output for traceability
PIPELINE_VERSION = "1.0"

## Section 3. Requirements 1, 2 and 3

**Rationale.** Each requirement is checked at the point where breaking it would otherwise cause a
confusing failure later. A wrong file name means the product is unknown; a missing tab means half the
tab cannot be built; a malformed `Model Group` means the output would be labelled for a product the
data does not support. Each check names the requirement, the file and what was expected.

One thing worth knowing about the supplied workbook: its `Raw Detailed Results` tab has
`Model Group` = `Grizzly Pouch`, which does not meet Requirement 3, while its `MasterModels TP-1D`
tab has `Grizzly XL YO`, which does. Those are different model families, the dual use runs and the
master models, so the product is taken from the master models tab, which is the one that carries the
convention, and a non conforming `Detailed Results` group is reported as a warning rather than
stopping the run. Change `STRICT_MODEL_GROUP` to `True` to make it stop instead.

In [ ]:
STRICT_MODEL_GROUP = False        # True makes a non conforming Detailed Results group an error


def norm_text(value):
    """Trim, then collapse runs of whitespace to single underscores. Non text passes through."""
    return re.sub(r"\s+", "_", value.strip()) if isinstance(value, str) else value


def same_label(a, b) -> bool:
    """Compare labels ignoring spacing, underscores, hyphens and case."""
    key = lambda s: re.sub(r"[\s_\-]+", "", str(s)).casefold()
    return key(a) == key(b)


def find_sheet(sheetnames, keys, workbook_name: str) -> str:
    """Match a tab on its letters and digits only, so spacing and TP-ID vs TP-1D do not matter."""
    for name in sheetnames:
        flat = re.sub(r"[^a-z0-9]", "", name.lower())
        if all(k in flat for k in keys):
            return name
    raise KeyError(f"Requirement 2 not met: {workbook_name} has no tab matching {keys}. "
                   f"Tabs found: {sheetnames}")


def split_model_group(model_group) -> tuple[str, str]:
    """Requirement 3: 'Grizzly_XL_YO' -> ('Grizzly_XL', 'YO'), the product and the initials."""
    tokens = [t for t in norm_text(str(model_group)).split("_") if t]
    if len(tokens) < 2:
        raise ValueError(f"Model Group is {model_group!r}, which does not meet Requirement 3. "
                         "It must read '<Product>_<your initials>', for example 'Grizzly_XL_YO'.")
    return "_".join(tokens[:-1]), tokens[-1]


def resolve_input(data_dir=None, filename=None) -> dict:
    """Find the workbook, check Requirement 1, and return the product and the date from its name."""
    data_dir = Path(data_dir or DATA_DIR)
    if not data_dir.is_dir():
        raise NotADirectoryError(f"DATA_DIR does not exist:\n  {data_dir}")

    if filename:
        candidates = [data_dir / filename]
        if not candidates[0].exists():
            raise FileNotFoundError(f"Input file not found:\n  {candidates[0]}")
    else:
        candidates = [p for p in sorted(data_dir.glob("Dual_Use_Calculations_TwoAge_B01_*.xlsx"))
                      if not p.name.startswith("~$")]     # skip Excel lock files

    matched = [(p, INPUT_REGEX.match(p.name)) for p in candidates]
    matched = [(p, m) for p, m in matched if m]
    if len(matched) != 1:
        raise FileNotFoundError(
            "Requirement 1 not met: expected exactly one file named "
            "'Dual_Use_Calculations_TwoAge_B01_<Product>_<Month>_<Day>_<Year>.xlsx' in "
            f"{data_dir}, found {len(matched)}"
            + (f" ({', '.join(p.name for p, _ in matched)})." if matched else "."))

    path, m = matched[0]
    return {"path": path, "product": norm_text(m["product"]),
            "source_date": f"{m['month']} {m['day']}, {m['year']}", "data_dir": data_dir}

## Section 4. Reading the two tabs

**Rationale.** The two tabs are different exports with different columns, so they are normalised into
one shape before anything looks anything up: the header is taken from row 2 because the DPM writes a
title in row 1, column names have their separators normalised, and `Gateway` and `Sex` are derived so
every later lookup is a three key match on gateway, sex and ERR rather than string matching on model
names.

`lookup_mean` raising when it does not find exactly one row is deliberate. A missing run and a
duplicated run both produce a wrong number silently otherwise, and a wrong number here propagates
into every interpolated cell, the chart and the tipping points.

In [ ]:
def read_tab(path, sheet: str) -> pd.DataFrame:
    """Read one DPM style tab: title in row 1, header in row 2, data below."""
    raw = pd.read_excel(path, sheet_name=sheet, header=1, dtype=object)
    raw.columns = [norm_text(str(c)) for c in raw.columns]
    for c in raw.columns:
        if raw[c].dtype == object:
            raw[c] = raw[c].map(lambda v: v.strip() if isinstance(v, str) else v)

    for need in ("Model_Group", "Model_Name", "Mortality_Model", "ERR", "Node", "Mean"):
        if need not in raw.columns:
            raise KeyError(f"Tab '{sheet}' is missing the required column "
                           f"'{need.replace('_', ' ')}'. Columns found: {list(raw.columns)}")

    raw = raw[raw["Model_Group"].notna()].copy()
    raw = raw[raw["Model_Group"].astype(str) != "Model Group"]      # any repeated header row
    raw["ERR"] = pd.to_numeric(raw["ERR"], errors="coerce")
    raw["Mean"] = pd.to_numeric(raw["Mean"], errors="coerce")

    name = raw["Model_Name"].astype(str)
    raw["Gateway"] = np.where(name.str.contains("G10"), "G10",
                              np.where(name.str.contains("G25"), "G25", None))
    raw["Sex"] = raw["Mortality_Model"].astype(str).map(
        lambda v: "Male" if "Male" in v else ("Female" if "Female" in v else None))

    if raw["Gateway"].isna().any():
        raise ValueError(f"{int(raw['Gateway'].isna().sum())} rows in '{sheet}' have neither "
                         "'G10' nor 'G25' in the model name.")
    if raw["Sex"].isna().any():
        raise ValueError(f"{int(raw['Sex'].isna().sum())} rows in '{sheet}' have a mortality "
                         "model that is neither male nor female.")
    return raw.reset_index(drop=True)


def load_inputs(resolved: dict) -> dict:
    """Read both tabs, then check Requirements 2 and 3."""
    path = resolved["path"]
    sheetnames = load_workbook(path, read_only=True).sheetnames

    names = {role: find_sheet(sheetnames, keys, path.name)
             for role, keys in SHEET_KEYS.items()}                     # Requirement 2
    frames = {role: read_tab(path, sheet) for role, sheet in names.items()}

    # Requirement 3, enforced on the master models tab, which is the one that carries the convention
    groups = frames["master"]["Model_Group"].dropna().unique()
    if len(groups) != 1:
        raise ValueError(f"'{names['master']}' holds {len(groups)} model groups ({list(groups)}). "
                         "This is the one product pipeline.")
    product, initials = split_model_group(groups[0])
    if not same_label(product, resolved["product"]):
        raise ValueError(f"Requirement 3 not met: the file name gives the product "
                         f"{resolved['product']!r} but Model Group in '{names['master']}' is "
                         f"{groups[0]!r}, giving {product!r}. Fix whichever is wrong.")

    det_groups = frames["detailed"]["Model_Group"].dropna().unique()
    for grp in det_groups:
        try:
            det_product, _ = split_model_group(grp)
            conforms = same_label(det_product, product)
        except ValueError:
            conforms = False
        if not conforms:
            message = (f"Model Group in '{names['detailed']}' is {grp!r}, which does not meet "
                       f"Requirement 3 for product {product!r}. These are the dual use runs, so "
                       "the product has been taken from the master models tab instead.")
            if STRICT_MODEL_GROUP:
                raise ValueError("Requirement 3 not met: " + message)
            warnings.warn(message)

    gateways = [g for g in GATEWAY_ORDER if g in set(frames["detailed"]["Gateway"])]
    if not gateways:
        raise ValueError(f"'{names['detailed']}' holds no G10 or G25 runs.")

    print(f"Product : {product}      Analyst initials: {initials}")
    print(f"  {names['detailed']:<24} {len(frames['detailed']):>3} rows   (dual use runs)")
    print(f"  {names['master']:<24} {len(frames['master']):>3} rows   (no quitting baseline)")
    print(f"  gateways: {', '.join(gateways)}")
    return {"frames": frames, "sheet_names": names, "product": product,
            "initials": initials, "gateways": gateways, **resolved}


def lookup_mean(df: pd.DataFrame, gateway: str, sex: str, err: float, label: str) -> pd.Series:
    """Exactly one row per gateway, sex and ERR, or the number that follows is meaningless."""
    hit = df[(df["Gateway"] == gateway) & (df["Sex"] == sex)
             & np.isclose(df["ERR"].astype(float), err)]
    if len(hit) != 1:
        raise ValueError(f"{label}: expected exactly one row for {gateway}, {sex}, ERR {err}, "
                         f"found {len(hit)}.")
    return hit.iloc[0]

## Section 5. The layout engine

**Rationale.** This is where the rule catalogue at the top becomes code. Every cell of the tab is
emitted as a `(row, column, value, style)` record by one function, so the layout can be read in one
place and the writer in Section 6 stays a dumb loop. Formulas are emitted as text so the output
workbook recalculates in Excel exactly as the source did, and the same numbers are computed in
Python alongside them, which is what Section 9 validates against.

Two places where the rebuild is deliberately more general than the source, both of which give
identical numbers on this workbook:

* The interpolation formula points at its own table's % dual use header row. Every formula in the
  source points at row 12, the first block's female header, which holds the same `0` to `1` vector,
  so the results agree; but it means the second block depends on a row twenty two rows above it for
  no reason, and moving a block would silently break it.
* The tipping point formula finds the bracketing pair from the values rather than having it typed in.
  The source has the pair hard coded per row and gets it right, including one row that crosses
  between 0.4 and 0.5 while the other three cross between 0.5 and 0.6. Finding it means new data
  does not need the formula re-pointed by hand.

In [ ]:
def block_rows(base: int) -> dict:
    """Every row number a block uses, derived from its first row."""
    return {
        "left_header": base + OFF["left_header"],
        "sex_label": {SEX_ORDER[0]: base + OFF["sex_label_1"], SEX_ORDER[1]: base + OFF["sex_label_2"]},
        "sex_first": {SEX_ORDER[0]: base + OFF["sex_rows_1"], SEX_ORDER[1]: base + OFF["sex_rows_2"]},
        "dual_header": {sex: base + OFF["dual_header"] + i * SEX_TABLE_STRIDE
                        for i, sex in enumerate(SEX_ORDER)},
        "nq_header": base + OFF["nq_header"],
        "nq_first": base + OFF["nq_first"],
    }


def build_difference_cells(detailed: pd.DataFrame, master: pd.DataFrame,
                           gateways: list[str]) -> tuple[list[tuple], dict]:
    """Emit every cell of the Difference tab. Returns (cells, computed values for validation)."""
    cells: list[tuple] = []          # (row, col, value, style)
    computed: dict = {}              # (row, col) -> number, the Python mirror of each formula
    add = lambda r, c, v, s=None: cells.append((r, c, v, s))

    tp_rows_data = []                # feeds the tipping point table after the last block

    for bi, gateway in enumerate(gateways):
        base = 1 + bi * BLOCK_STRIDE
        pos = block_rows(base)

        # ---------------------------------------------------- 1. left table
        for i, head in enumerate(["Model Name", "ERR", "Node", "Mean"], start=1):
            add(pos["left_header"], i, head, "header")
        add(pos["left_header"], 5, GATEWAY_FRACTION[gateway], "gateway_pct")
        add(pos["left_header"], 6, SEX_ORDER[0], "gateway")

        left_row = {}
        for sex in SEX_ORDER:
            add(pos["sex_label"][sex], 1, sex, "sexlabel")
            r = pos["sex_first"][sex]
            for err in ERR_A_VALUES + ERR_B_VALUES:
                rec = lookup_mean(detailed, gateway, sex, err, "Detailed Results")
                add(r, 1, rec["Model_Name"])
                add(r, 2, err)
                add(r, 3, rec["Node"])
                add(r, 4, float(rec["Mean"]), "count")
                left_row[(sex, err)] = r
                computed[(r, 4)] = float(rec["Mean"])
                r += 1

        # ------------------------------------- 2. dual use interpolation tables
        interp_row = {}
        for sex in SEX_ORDER:
            hdr = pos["dual_header"][sex]
            add(hdr - 2, 6, sex, "bold")                       # 'Male' / 'Female'
            add(hdr - 1, COL["H"], "% dual use", "bold")
            add(hdr, 6, "ERR A", "bold")
            add(hdr, 7, "ERR B", "bold")
            for i, w in enumerate(DUAL_USE_STEPS):
                add(hdr, COL["H"] + i, w, "header_plain")

            r = hdr + 1
            for err_a in ERR_A_VALUES:
                add(r, 6, err_a)                               # written once per pair
                for err_b in ERR_B_VALUES:
                    add(r, 7, err_b)
                    ra, rb = left_row[(sex, err_a)], left_row[(sex, err_b)]
                    va, vb = computed[(ra, 4)], computed[(rb, 4)]

                    add(r, COL["H"], f"=D{ra}", "count")       # w = 0, the exclusive use mean
                    add(r, COL["R"], f"=D{rb}", "count")       # w = 1, the dual use mean
                    computed[(r, COL["H"])] = va
                    computed[(r, COL["R"])] = vb
                    for i in range(1, len(DUAL_USE_STEPS) - 1):
                        col = COL["H"] + i
                        letter = L(col)
                        add(r, col, f"=(1-{letter}${hdr})*$H{r}+{letter}${hdr}*$R{r}", "count")
                        computed[(r, col)] = (1 - DUAL_USE_STEPS[i]) * va + DUAL_USE_STEPS[i] * vb

                    interp_row[(sex, err_a, err_b)] = r
                    r += 1
                r += 1                                          # blank row between the ERR A pairs

        # ------------------------------------------- 3. no quitting comparison
        add(pos["nq_header"], 3, "No quitting", "header_plain")
        add(pos["nq_header"], 4, "% change", "header_plain")
        r = pos["nq_first"]
        for sex in SEX_ORDER:
            add(r, 2, sex[0], "underline")                      # 'M' / 'F'
            r += 1
            for err in ERR_A_VALUES:
                mrec = lookup_mean(master, gateway, sex, err, "MasterModels TP-ID")
                lr = left_row[(sex, err)]
                add(r, 2, err, "header_plain")
                add(r, 3, float(mrec["Mean"]), "count")
                add(r, 4, f"=(D{lr}-C{r})/D{lr}", "pct")
                computed[(r, 4)] = (computed[(lr, 4)] - float(mrec["Mean"])) / computed[(lr, 4)]
                r += 1

        # ----------------------------------------------------- 4. chart feed
        add(pos["dual_header"][SEX_ORDER[0]], COL["T"], "Pasted from left before transposing", "bold")
        add(pos["dual_header"][SEX_ORDER[0]], COL["AF"], "Transposed for tables", "bold")
        for pi, (err_a, err_b) in enumerate(MATCHED_PAIRS):
            src_row = interp_row[(SEX_ORDER[0], err_a, err_b)]
            series = [computed[(src_row, COL["H"] + i)] for i in range(len(DUAL_USE_STEPS))]
            for i, v in enumerate(series):                       # across, as values
                add(src_row, COL["T"] + i, v, "count")
            for i, v in enumerate(series):                       # down, transposed for the chart
                add(pos["dual_header"][SEX_ORDER[0]] + 1 + i, COL["AF"] + pi, v, "count")

        # ------------------------------------------------- 5. tipping points
        for sex in TP_SEXES:
            hdr = pos["dual_header"][sex]
            for err_a, err_b in MATCHED_PAIRS:
                r = interp_row[(sex, err_a, err_b)]
                series = [computed[(r, COL["H"] + i)] for i in range(len(DUAL_USE_STEPS))]
                tp_rows_data.append({"sex": sex, "gateway": gateway, "err_a": err_a,
                                     "err_b": err_b, "row": r, "hdr": hdr, "series": series})

    # The tipping point table sits below the last block
    last_base = 1 + (len(gateways) - 1) * BLOCK_STRIDE
    tp_start = last_base + OFF["tp_offset"]
    add(tp_start, 10, "Tipping points", "bold")
    for col, head in ((11, "ERR A/ERR B"), (12, "Gateway"),
                      (13, "Used Product in Combination with Cigarettes"),
                      (14, "Used Product Exclusively")):
        add(tp_start + 1, col, head, "underline_bold")

    r = tp_start + 2
    last_gateway = None
    for rec in tp_rows_data:
        if last_gateway is not None and rec["gateway"] != last_gateway:
            r += 1                                              # blank row between gateways
        last_gateway = rec["gateway"]

        # Find the pair of % dual use steps that brackets the sign change
        series = rec["series"]
        cross = next((i for i in range(len(series) - 1)
                      if series[i] > 0 >= series[i + 1]), None)
        add(r, 10, rec["sex"])
        add(r, 11, f"{rec['err_a']:.2f}/{rec['err_b']:.2f}")
        add(r, 12, rec["gateway"])
        if cross is None:
            add(r, 13, "no crossing")
            add(r, 14, "no crossing")
        else:
            lo, hi = L(COL["H"] + cross), L(COL["H"] + cross + 1)
            # Rounded and formatted with :g, otherwise 0.6 - 0.5 writes 0.09999999999999998
            # into the formula text
            step = round(DUAL_USE_STEPS[cross + 1] - DUAL_USE_STEPS[cross], 10)
            add(r, 13, f"={lo}${rec['hdr']}+{step:g}*{lo}{rec['row']}/"
                       f"({lo}{rec['row']}+ABS({hi}{rec['row']}))", "pct")
            add(r, 14, f"=1-M{r}", "pct")
            v_lo, v_hi = series[cross], series[cross + 1]
            tp = DUAL_USE_STEPS[cross] + step * v_lo / (v_lo + abs(v_hi))
            computed[(r, 13)] = tp
            computed[(r, 14)] = 1 - tp
        r += 1

    return cells, computed

## Section 6. Writing `Difference_calc`

**Rationale.** A dumb writer over the records from Section 5, so the layout has exactly one
definition. The styles reproduce the source: an orange header fill, blue sex labels, a yellow gateway
cell, thousands separators on counts and `0%` on the percentages, and the merged title over the
tipping point table.

In [ ]:
STYLES = {
    "header":         dict(bold=True, fill=FILL_HEADER, border="thick"),
    "header_plain":   dict(fill=FILL_HEADER, border="thick"),
    "underline_bold": dict(bold=True, border="thick"),
    "underline":      dict(border="thick"),
    "bold":           dict(bold=True),
    "sexlabel":       dict(fill=FILL_SEX, border="thick"),
    "gateway":        dict(fill=FILL_GATEWAY),
    "gateway_pct":    dict(fill=FILL_GATEWAY, fmt=FMT_PCT),
    "count":          dict(bold=True, fmt=FMT_COUNT),
    "pct":            dict(bold=True, fmt=FMT_PCT),
}
COLUMN_WIDTHS = {1: 67, 2: 12.5, 3: 12.5, 4: 12, 6: 21, 7: 10, 11: 13, 12: 10, 13: 42, 14: 24}


def write_difference_sheet(ws, cells: list[tuple], tp_title_row: int = None) -> None:
    """Write every emitted cell with its style."""
    thick = Side(style="thick", color="000000")
    for row, col, value, style in cells:
        cell = ws.cell(row=row, column=col, value=value)
        spec = STYLES.get(style, {})
        cell.font = Font(name=FONT, size=FONT_SIZE, bold=spec.get("bold", False))
        if spec.get("fill"):
            cell.fill = PatternFill("solid", fgColor=spec["fill"])
        if spec.get("border") == "thick":
            cell.border = Border(bottom=thick)
        if spec.get("fmt"):
            cell.number_format = spec["fmt"]

    for col, width in COLUMN_WIDTHS.items():
        ws.column_dimensions[L(col)].width = width
    if tp_title_row:
        ws.merge_cells(start_row=tp_title_row, start_column=10,
                       end_row=tp_title_row, end_column=14)
        ws.cell(row=tp_title_row, column=10).alignment = Alignment(horizontal="left")


def write_source_tab(ws, frame: pd.DataFrame, original_columns: list[str],
                     title: str = "Detailed Model Results") -> None:
    """Carry a source tab through in its original shape: title row, header row, then the data."""
    ws.cell(row=1, column=1, value=title).font = Font(name=FONT, size=FONT_SIZE, bold=True)

    # pandas names a headerless column 'Unnamed: N'. The DPM leaves the upper half of the
    # posterior interval pair headerless, so that placeholder is written back as blank.
    headers = ["" if str(c).startswith("Unnamed") else str(c) for c in original_columns]
    for j, col in enumerate(headers, start=1):
        cell = ws.cell(row=2, column=j, value=col)
        cell.font = Font(name=FONT, size=FONT_SIZE, bold=True)
        cell.fill = PatternFill("solid", fgColor=FILL_HEADER)

    for i, rec in enumerate(frame.itertuples(index=False), start=3):
        for j, value in enumerate(rec, start=1):
            blank = value is None or (isinstance(value, float) and np.isnan(value))
            cell = ws.cell(row=i, column=j, value=None if blank else value)
            cell.font = Font(name=FONT, size=FONT_SIZE)
            if headers[j - 1] in ("Mean", "95% PI"):
                cell.number_format = FMT_COUNT

    for j, col in enumerate(headers, start=1):
        # An entirely empty column gives NaN from .max(), so fall back to the header width
        longest = frame.iloc[:, j - 1].astype(str).str.len().max() if len(frame) else 0
        longest = 0 if pd.isna(longest) else int(longest)
        ws.column_dimensions[L(j)].width = min(max(len(col) + 2, longest + 2), 60)
    ws.freeze_panes = "A3"

## Section 7. The output workbook

**Rationale.** A new workbook rather than an edit of the input, so the input is never at risk and the
output is reproducible from scratch every run. It carries exactly the three tabs the requirement
asks for: the two sources under the names they had in the input, and `Difference_calc`.

The source tabs are written from the values read in Section 4, so what the rebuild used and what the
output shows are the same numbers by construction.

In [ ]:
def build_output_workbook(loaded: dict, cells: list[tuple], tp_title_row: int,
                          out_dir=None, stamp: str = None) -> Path:
    """Write the three tab output workbook and return its path."""
    out_dir = Path(out_dir) if out_dir else loaded["data_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)
    stamp = stamp or datetime.now().strftime(OUTPUT_DATE_FORMAT)
    path = out_dir / OUTPUT_TEMPLATE.format(product=loaded["product"], stamp=stamp)

    wb = Workbook()
    wb.remove(wb.active)

    for role in ("detailed", "master"):
        sheet_name = loaded["sheet_names"][role]
        frame = loaded["raw_frames"][role]
        ws = wb.create_sheet(sheet_name[:31])
        write_source_tab(ws, frame, list(frame.columns))

    ws = wb.create_sheet(OUTPUT_DIFF_SHEET)
    write_difference_sheet(ws, cells, tp_title_row=tp_title_row)

    wb.save(path)
    print(f"File saved: {path.name}")
    print(f"  tabs: {', '.join(wb.sheetnames)}")
    print(f"  folder: {path.parent}")
    return path

## Section 8. Orchestrator and run

**Rationale.** One call, so there is no order to remember. It returns the emitted cells and the
computed mirror, which Section 9 needs, plus the frames, so anything can be inspected afterwards
without rerunning.

In [ ]:
def run_pipeline(data_dir=None, out_dir=None, filename=None) -> dict:
    """Resolve the input, check all three requirements, rebuild the tab, write the workbook."""
    resolved = resolve_input(data_dir or DATA_DIR, filename or INPUT_FILE)
    loaded = load_inputs(resolved)

    # The output carries the source tabs exactly as read, before the derived columns were added
    loaded["raw_frames"] = {
        role: pd.read_excel(resolved["path"], sheet_name=loaded["sheet_names"][role], header=1)
        for role in ("detailed", "master")}

    cells, computed = build_difference_cells(loaded["frames"]["detailed"],
                                             loaded["frames"]["master"],
                                             loaded["gateways"])
    last_base = 1 + (len(loaded["gateways"]) - 1) * BLOCK_STRIDE
    tp_title_row = last_base + OFF["tp_offset"]

    path = build_output_workbook(loaded, cells, tp_title_row,
                                 out_dir=out_dir or OUT_DIR)

    print(f"  {OUTPUT_DIFF_SHEET}: {len(cells)} cells written, "
          f"{sum(1 for _, _, v, _ in cells if isinstance(v, str) and v.startswith('='))} formulas")
    return {"loaded": loaded, "cells": cells, "computed": computed, "output_file": path,
            "tp_title_row": tp_title_row, "product": loaded["product"],
            "initials": loaded["initials"]}


result = run_pipeline()

In [ ]:
# The rebuilt tipping points, the headline output of the tab
tp_start = result["tp_title_row"] + 2
rows = []
for (r, c), v in sorted(result["computed"].items()):
    if c == 13 and r >= tp_start:
        cells = {cc: vv for (rr, cc, vv, _) in
                 [(a, b, d, e) for a, b, d, e in result["cells"] if a == r]}
        rows.append({"Sex": cells.get(10), "ERR A/ERR B": cells.get(11), "Gateway": cells.get(12),
                     "In combination with cigarettes": f"{v:.1%}",
                     "Exclusively": f"{result['computed'][(r, 14)]:.1%}"})
display(pd.DataFrame(rows))

## Section 9. Validation

**Rationale.** The point of the rebuild is that it agrees with the hand built tab, so this compares
them cell by cell rather than taking it on trust. When the input workbook still carries its original
`Difference` tab, every interpolated value, every `% change` and every tipping point is checked
against the values Excel had cached there.

One expected difference, and it is worth understanding rather than loosening the tolerance blindly.
The `% change` column divides by the no quitting baseline, and the `MasterModels TP-ID` tab stores
that baseline rounded to whole survivors (`2679`), while the original `Difference` tab was built from
the unrounded export value (`2678.813451`). Rebuilding from the tab as required therefore lands
within about 0.02 of a percentage point, and both display as the same figure at the `0%` format the
column uses. Everything else must match exactly.

In [ ]:
def validate_against_source(result: dict, atol_interp: float = 1e-6,
                            atol_pct: float = 5e-4) -> pd.DataFrame:
    """Compare every rebuilt value with the cached value in the source Difference tab."""
    path = result["loaded"]["path"]
    sheetnames = load_workbook(path, read_only=True).sheetnames
    if SOURCE_DIFF_SHEET not in sheetnames:
        print(f"No '{SOURCE_DIFF_SHEET}' tab in the input, so there is nothing to compare against.")
        return pd.DataFrame()

    src = load_workbook(path, data_only=True)[SOURCE_DIFF_SHEET]
    checks = []
    for (row, col), mine in sorted(result["computed"].items()):
        theirs = src.cell(row=row, column=col).value
        if theirs is None or isinstance(theirs, str):
            continue
        kind = ("% change" if col == 4 and not isinstance(mine, (int, np.integer))
                and row % BLOCK_STRIDE in (16 % BLOCK_STRIDE, 17 % BLOCK_STRIDE,
                                           19 % BLOCK_STRIDE, 20 % BLOCK_STRIDE)
                else "tipping point" if col in (13, 14) and row >= result["tp_title_row"]
                else "left table" if col == 4
                else "interpolation")
        checks.append({"cell": f"{L(col)}{row}", "kind": kind, "rebuilt": float(mine),
                       "source": float(theirs), "diff": abs(float(mine) - float(theirs))})

    report = pd.DataFrame(checks)
    if report.empty:
        print("The source Difference tab holds no cached values to compare against.")
        return report

    tol = report["kind"].map(lambda k: atol_pct if k == "% change" else atol_interp)
    report["ok"] = report["diff"] <= tol
    summary = report.groupby("kind").agg(cells=("ok", "size"), matched=("ok", "sum"),
                                         worst=("diff", "max")).reset_index()
    print(summary.to_string(index=False))

    failures = report[~report["ok"]]
    if len(failures):
        print("\nCells outside tolerance:")
        print(failures.to_string(index=False))
        raise AssertionError(f"{len(failures)} rebuilt cell(s) do not match the source tab.")
    print(f"\nAll {len(report)} comparable cells match the source Difference tab.")
    return report


report = validate_against_source(result)

In [ ]:
# Structural checks on the written workbook, which do not depend on a source tab being present
wb = load_workbook(result["output_file"])
expected_tabs = [result["loaded"]["sheet_names"]["detailed"][:31],
                 result["loaded"]["sheet_names"]["master"][:31], OUTPUT_DIFF_SHEET]
assert wb.sheetnames == expected_tabs, f"Tabs are {wb.sheetnames}, expected {expected_tabs}"

ws = wb[OUTPUT_DIFF_SHEET]
n_gateways = len(result["loaded"]["gateways"])

# Every block must carry its full left table, four interpolation tables and the no quitting rows
for bi in range(n_gateways):
    base = 1 + bi * BLOCK_STRIDE
    pos = block_rows(base)
    assert ws.cell(row=pos["left_header"], column=1).value == "Model Name"
    assert ws.cell(row=pos["left_header"], column=5).value in GATEWAY_FRACTION.values()
    for sex in SEX_ORDER:
        hdr = pos["dual_header"][sex]
        assert [ws.cell(row=hdr, column=COL["H"] + i).value for i in range(11)] == DUAL_USE_STEPS
    assert ws.cell(row=pos["nq_header"], column=3).value == "No quitting"

# The formula count is fixed by the geometry: 11 per interpolation row, 4 per no quitting table,
# and 2 per tipping point row
n_formulas = sum(1 for _, _, v, _ in result["cells"] if isinstance(v, str) and v.startswith("="))
expected = n_gateways * (4 * 2 * 11 + 4) + len(TP_SEXES) * n_gateways * len(MATCHED_PAIRS) * 2
assert n_formulas == expected, f"{n_formulas} formulas written, expected {expected}"

print(f"Workbook checks passed: {len(wb.sheetnames)} tabs, {n_gateways} gateway block(s), "
      f"{n_formulas} live formulas in {OUTPUT_DIFF_SHEET}.")

## Section 10. Troubleshooting and known differences

### Errors

| Message | Cause and fix |
|---|---|
| `Requirement 1 not met: expected exactly one file named ...` | The workbook is missing, misnamed, or there are several matches in the folder. The name must be `Dual_Use_Calculations_TwoAge_B01_<Product>_<Month>_<Day>_<Year>.xlsx`. Name one explicitly with `INPUT_FILE` if the folder holds more than one. |
| `Requirement 2 not met: ... has no tab matching ...` | A `Detailed Results` or `MasterModels TP-ID` tab is missing. Matching ignores case, spacing and punctuation, so `Raw Detailed Results` and `MasterModels TP-1D` are both fine, but the words themselves must be there. |
| `Model Group is '...', which does not meet Requirement 3` | It must read `<Product>_<your initials>`, for example `Grizzly_XL_YO`. |
| `Requirement 3 not met: the file name gives the product ...` | The file name and `Model Group` disagree about the product. Fix whichever is wrong. |
| `Model Group in '...' is '...', which does not meet Requirement 3` warning | The dual use runs sit in their own model group, `Grizzly Pouch` in the supplied workbook. The product comes from the master models tab. Set `STRICT_MODEL_GROUP = True` to make this an error. |
| `expected exactly one row for G10, Male, ERR 0.05, found 0` | An ERR is missing from an export. All four of 0.05, 0.10, 1.05 and 1.10 are needed per gateway and sex in `Detailed Results`, and both of 0.05 and 0.10 in `MasterModels TP-ID`. |
| `found 2` on the same message | A model was run twice and the export holds both. Delete the duplicate. |
| `rows have neither 'G10' nor 'G25' in the model name` | A gateway cannot be identified, so the row cannot be placed in a block. |
| `is missing the required column '...'` | A tab was edited and a column renamed or removed. |
| `N rebuilt cell(s) do not match the source tab` | Section 9 prints the offending cells with both values. A difference in the `% change` rows is explained below; anywhere else means the inputs and the source tab were not built from the same numbers. |
| `Permission denied` when writing | The output workbook is open in Excel. Close it and rerun. |

### Deliberate differences from the source tab

| Difference | Why |
|---|---|
| The interpolation formula points at its own table's header row, not always row 12 | Every formula in the source points at the first block's female header row, which happens to hold the same `0` to `1` vector, so the numbers agree. Pointing each table at its own header removes a cross block dependency that would break silently if a block moved. |
| The tipping point bracketing pair is found from the values | The source has it typed in per row. It is correct there, including one row that crosses between 0.4 and 0.5 while the others cross between 0.5 and 0.6, but new data would need every formula re-pointed by hand. |
| `% dual use` labels appear on all four tables, and the chart feed labels on both blocks | The source omits them on the second block's female table and on the second block entirely. Nothing depends on them; they are captions. |
| The chart feed values keep full precision | The source pasted them rounded to whole survivors. Displayed with the same `#,##0` format they read identically. |
| `% change` can differ by up to about 0.02 of a percentage point | The rebuild divides by the no quitting baseline as stored in `MasterModels TP-ID`, which is rounded to whole survivors. The original was built from the unrounded export. Both display the same at `0%`. Requirement 2 says the third tab comes from the first two, so the tab as given is what is used. |
| The output is a new three tab workbook | The input is never modified, and the output is reproducible from scratch on every run. The source `Chart1` chartsheet is not carried over, since it points at the original `Difference` tab; rebuild it against `Difference_calc` columns `AF` and `AG` if it is needed. |